# Purpose of this script is to perform Sequential outer merge on ('iso3', 'Year')

In [21]:
import numpy as numpy
import pandas as pd

In [22]:
# Import processed data to merge
Gini_df = pd.read_parquet("../Data/Processed/Parquet_data/Gini_Index.parquet")
Gdp_df = pd.read_parquet("../Data/Processed/Parquet_data/Gdp_percap(2021 usd).parquet")
Top20_df = pd.read_parquet("../Data/Processed/Parquet_data/Income_Share(Top20).parquet")
Bottom50_df = pd.read_parquet("../Data/Processed/Parquet_data/Income_Share(Bottom50%).parquet")

In [23]:
# Print column names to inspect
print(Gini_df.columns.tolist())
print(Gdp_df.columns.tolist())
print(Top20_df.columns.tolist())
print(Bottom50_df.columns.tolist())

['country', 'year', 'gini_disp', 'gini_disp_se', 'gini_mkt', 'gini_mkt_se', 'abs_red', 'abs_red_se', 'rel_red', 'rel_red_se', 'region', 'iso3', 'abs_red_source']
['iso3', 'Country', 'year', 'pop', 'rgdpe', 'region', 'gdp_per_capita']
['Country', 'Variable', 'Percentile', 'Year', 'Value', 'Data quality', 'iso3', 'region']
['Country', 'Variable', 'Percentile', 'Year', 'Value', 'Data quality', 'iso3', 'region']


In [24]:
# Changing the column names to avoid complication while merging the datasets
Gini_df = Gini_df.rename(columns={'year': 'Year', 'country': 'Country'})
Gdp_df = Gdp_df.rename(columns={'year':'Year'})
Top20_df = Top20_df.rename(columns={'Variable':'Variable_Top20', 'Percentile':'Percentile_Top20', 'Value':'Value_Top20', 
                                    'Data quality':'DataQuality_Top20'})
Bottom50_df = Bottom50_df.rename(columns={'Variable':'Variable_Bottom50', 'Percentile':'Percentile_Bottom50', 'Value':'Value_Bottom50', 
                                    'Data quality':'DataQuality_Bottom50'})

In [13]:
# -----------------------------------------------------------------------
# Coverage diff -- see which countries are missing from which
# source BEFORE merging, so gaps are understood rather than discovered
# later as mystery NaNs.
# -----------------------------------------------------------------------
datasets = {
     'GiniIndex':Gini_df, 'GdpPerCapita':Gdp_df, 'IncomeShare(Top20)':Top20_df, 'IncomeShare(Bottom50)':Bottom50_df
 }
print("\n=== Country (iso3) coverage per source ===")
country_sets = {name: set(df["iso3"].unique()) for name, df in datasets.items()}
for name, s in country_sets.items():
    print(f"{name:10s}: {len(s)} unique countries")
 
all_countries = set.union(*country_sets.values())
print(f"\nUnion across all sources: {len(all_countries)} countries")
 
print("\n=== Countries missing from each source ===")
for name, s in country_sets.items():
    missing = sorted(all_countries - s)
    if missing:
        print(f"\nMissing from {name} ({len(missing)}):")
        print(missing)


=== Country (iso3) coverage per source ===
GiniIndex : 114 unique countries
GdpPerCapita: 112 unique countries
IncomeShare(Top20): 114 unique countries
IncomeShare(Bottom50): 114 unique countries

Union across all sources: 114 countries

=== Countries missing from each source ===

Missing from GdpPerCapita (2):
['PRI', 'TON']


In [ ]:
# Country/region are static per iso3 (they don't change by year), and all
# four dataframes carry their own copy of these columns. Build a single
# clean lookup table (one row per country) instead of letting duplicate
# copies ride along into the merge. This also avoids ending up with
# multiple slightly different names for the same country (e.g. "USA" vs
# "United States") sitting side by side once the datasets are combined --
# by collapsing to one row per iso3, the panel keeps a single consistent
# name/region per country instead of a version per source.
METADATA_COLS = ['Country', 'region']

# For each dataframe, keep only iso3 + metadata columns, and collapse
# each country's repeated per-year rows down to a single row.
metadata_frames = [df[['iso3'] + METADATA_COLS].drop_duplicates() for df in datasets.values()]

# Stack all four sources' metadata on top of each other, then deduplicate
# again across sources -- keeping exactly one row per iso3 overall.
country_lookup = pd.concat(metadata_frames, ignore_index=True).drop_duplicates(subset='iso3')
print(country_lookup)

In [17]:
# Merge all four datasets on (iso3, Year), with Country/region dropped
# from each one first -- they're already saved separately in
# country_lookup, so keeping them here would cause duplicate-column
# collisions once more than two dataframes are merged in sequence.
panel = None
for name, df in datasets.items():
    df = df.drop(columns=['Country', 'region'])
    if panel is None:
        # First dataframe: just becomes the starting point for panel.
        panel = df.copy()
    else:
        # Outer join keeps every country-year row from every source --
        # if a source doesn't cover that year, the cell becomes NaN
        # instead of the row being dropped entirely.
        panel = panel.merge(df, on=['iso3', 'Year'], how='outer')

# Attach Country/region back onto the panel, joining on iso3 ALONE (not
# Year)
panel = panel.merge(country_lookup, on='iso3', how='left')

# Sort rows so each country's years run in order, then reset the index
# to a clean 0, 1, 2... sequence 
panel = panel.sort_values(['iso3', 'Year']).reset_index(drop=True)

In [20]:
panel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7618 entries, 0 to 7617
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Year                  7618 non-null   int64  
 1   gini_disp             5155 non-null   float64
 2   gini_disp_se          5155 non-null   float64
 3   gini_mkt              5155 non-null   float64
 4   gini_mkt_se           5155 non-null   float64
 5   abs_red               5155 non-null   float64
 6   abs_red_se            2633 non-null   float64
 7   rel_red               5155 non-null   float64
 8   rel_red_se            2633 non-null   float64
 9   iso3                  7618 non-null   object 
 10  abs_red_source        5155 non-null   object 
 11  pop                   7187 non-null   float64
 12  rgdpe                 7187 non-null   float64
 13  gdp_per_capita        7187 non-null   float64
 14  Variable_Top20        5698 non-null   object 
 15  Percentile_Top20     